## Load the dataset:

In [1]:
from datasets import Dataset


dataset = Dataset.from_file("../datasets/sw-en/train/data-00000-of-00001.arrow")
# dataset.save_to_disk("../datasets/eng")

c:\Users\EthanK\.conda\envs\culturaai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset

Dataset({
    features: ['prompt', 'input', 'output'],
    num_rows: 1115700
})

In [3]:
from transformers import AutoTokenizer
model_path ='../models/umt5-base'
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only= True)


Split Dataset


In [4]:
from datasets import  DatasetDict

dataset = dataset.train_test_split(test_size=0.1, seed =42)

dataset = DatasetDict({
    "train": dataset["train"],
    "validation" : dataset["test"]
    
})

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'input', 'output'],
        num_rows: 1004130
    })
    validation: Dataset({
        features: ['prompt', 'input', 'output'],
        num_rows: 111570
    })
})

PreProcessing

In [6]:
def preprocess_function(batch):
    inputs = [ p + ": " + i for p, i in zip(batch["prompt"],batch["input"])]
    targets = batch["output"]
    
    # Tokenize the inputs
    model_inputs = tokenizer(
        targets,
        max_length =128,
        truncation = True,
        padding = "max_length"
    )
    
    # Tokenize the targets (labels)
    labels = tokenizer(
        targets, 
        max_length=128, 
        truncation=True, 
        padding="max_length").input_ids

    model_inputs["labels"] = labels
    return model_inputs

    

Apply Preprocessing:

In [7]:
tokenized_dataset =dataset.map(
    preprocess_function,
    batched = True,
    remove_columns=["prompt", "input", "output"]
)

In [8]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1004130
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 111570
    })
})

## Load Model and Training Set_Up

In [12]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)


cuda


In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch
model_path = '../models/umt5-base'

# Load model (umt5-base)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path, local_files_only = True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# Training configuration
training_args = Seq2SeqTrainingArguments(
    output_dir="../models/results",
    evaluation_strategy="steps",
    eval_steps=100,
    logging_steps=1,          # 🔥 Log every step
    save_steps=500,
    save_total_limit=2,
    learning_rate=3e-5,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    warmup_steps=100,
    predict_with_generate=True,
    fp16=False,
    bf16=False,
    dataloader_num_workers=2,
    report_to="tensorboard"
)

Trainer: 


In [ ]:
trainer = Seq2SeqTrainer(
    model = model,
    args= training_args,
    train_dataset= tokenized_dataset["train"],
    eval_dataset = tokenized_dataset["validation"],
    tokenizer = tokenizer
)
from transformers.trainer_callback import PrinterCallback

# Add this line before trainer.train()
trainer.remove_callback(PrinterCallback)  # Remove default printer
trainer.add_callback(PrinterCallback())

c:\Users\EthanK\.conda\envs\culturaai\lib\site-packages\accelerate\accelerator.py:451: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Training:

In [ ]:
import time

start = time.time()
trainer.train()
print(f"Training took: {(time.time()-start)/60:.2f} minutes")


  0%|          | 0/188274 [00:00<?, ?it/s]

KeyboardInterrupt: 

Saving the Model

In [ ]:
trainer.save_model("../models/umt5-en-sw")
tokenizer.save_pretrained("../models/umt5-en-sw")

# Load pipeline for inference
from transformers import pipeline

translator = pipeline("translation", model="../models/umt5-en-sw", tokenizer="../models/umt5-en-sw")

print(translator("Translate the following text to Swahili: I love learning AI"))
print(translator("Translate the following text to English: Napenda kula matunda."))


In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
